In [13]:
!pip install torch transformers datasets sentencepiece tqdm scipy sentence-transformers datasets evaluate scikit-learn evaluate wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=3cb73501cb9ce1ee7faf97b71e82b7401d4176eb542b917e34ed38d08502113e
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [2]:
from sentence_transformers import SentenceTransformer, models, losses, InputExample, evaluation
import torch
import os
from datasets import load_dataset
import numpy as np
from typing import Tuple
from torch import nn
from datasets import load_dataset, DatasetDict, load_from_disk # Import load_from_disk
from sentence_transformers.readers import STSBenchmarkDataReader
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
import evaluate

In [3]:
import random
from typing import List, Tuple, Optional
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm
import numpy as np
from scipy.stats import spearmanr
import pandas as pd

class SentenceEmbedder(nn.Module):
    """
    BERT-based sentence embedder with mean-pooling or cls pooling.
    The model is built so passing the same input twice with dropout yields different embeddings
    (SimCSE unsupervised trick).

    We explicitly separate:
      - the encoder (self.enc): maps tokens -> hidden states
      - the pooler/projection (self.projection): maps hidden states -> low-dim sentence embedding

    Following Wang et al. (2023), the performance loss in low-dimensional settings can be
    understood as the sum of:
      - Performance Loss of the Encoder
      - Performance Loss of the Pooler

    This motivates training schemes where we freeze one part and optimize the other.
    """
    def __init__(self, model_name='bert-base-uncased', pooling: str = 'mean',
                 device='cuda', proj_size: Optional[int] = None):
        super().__init__()
        assert pooling in ('mean', 'cls')
        self.device = device
        self.enc = AutoModel.from_pretrained(model_name)
        self.pooling = pooling
        hidden_size = self.enc.config.hidden_size

        # Projection head = "pooler" in the Wang et al. sense
        if proj_size is None or proj_size == hidden_size:
            # No separate projection if proj_size is None or matches hidden_size
            self.projection = nn.Identity()
            self.proj_size = hidden_size
        else:
            self.projection = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),  # Optional intermediate layer
                nn.Tanh(),                            # Or ReLU or other activation
                nn.Linear(hidden_size, proj_size)
            )
            self.proj_size = proj_size

        self.to(device)

    # --- NEW: convenience methods to (un)freeze encoder vs pooler ---

    def freeze_encoder(self):
        for p in self.enc.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.enc.parameters():
            p.requires_grad = True

    def freeze_pooler(self):
        for p in self.projection.parameters():
            p.requires_grad = False

    def unfreeze_pooler(self):
        for p in self.projection.parameters():
            p.requires_grad = True

    def forward(self, input_ids, attention_mask, return_encoder_output: bool = False):
        """
        Encode batch of tokenized sentences and return normalized embeddings.

        Args:
            input_ids, attention_mask: tensors (batch, seq_len)
            return_encoder_output:
                - False (default): return final pooled+projected embedding (pooler output)
                - True: return a tuple (encoder_embedding, pooler_embedding)
                  where encoder_embedding is the pooled encoder output before projection,
                  and pooler_embedding is the final low-dim embedding.

        Returns:
            If return_encoder_output is False:
                z: L2-normalized embeddings (batch, proj_size)
            If True:
                (enc_norm, z): both L2-normalized, shapes (batch, H) and (batch, proj_size)
        """
        outputs = self.enc(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
            output_hidden_states=True
        )
        last_hidden = outputs.last_hidden_state  # (B, L, H)

        if self.pooling == 'mean':
            # mean pooling over tokens with attention mask
            mask = attention_mask.unsqueeze(-1).type_as(last_hidden)  # (B, L, 1)
            summed = (last_hidden * mask).sum(1)  # (B, H)
            denom = mask.sum(1).clamp(min=1e-9)
            pooled = summed / denom               # encoder-level sentence embedding
        elif self.pooling == 'cls':
            pooled = last_hidden[:, 0]
        else:
            raise ValueError(f"Unknown pooling mode: {self.pooling}")

        z = self.projection(pooled)
        z = F.normalize(z, p=2, dim=1)

        if return_encoder_output:
            enc_norm = F.normalize(pooled, p=2, dim=1)
            return enc_norm, z

        return z

    def get_sentence_embedding_dimension(self):
        """Returns the dimension of the final sentence embeddings after pooling and projection."""
        return self.proj_size

In [4]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


# load model

In [6]:
import json, os, torch
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load_dir = "/content/drive/MyDrive/weights-UML-train/bert_proj128_scientific"  # CHANGE

# 1) load config
with open(os.path.join(load_dir, "model_config.json"), "r") as f:
    cfg = json.load(f)

# 2) tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg["base_model_name"])

# 3) model skeleton
model = SentenceEmbedder(
    model_name=cfg["base_model_name"],
    pooling=cfg.get("pooling", "mean"),
    proj_size=cfg["proj_size"],
    device=device,
)

# 4) weights
state_dict = torch.load(os.path.join(load_dir, "pytorch_model.bin"),
                        map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print("Loaded model with dim =", model.get_sentence_embedding_dimension())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loaded model with dim = 128


# helpers

In [38]:
import numpy as np
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch

def encode_sentences(
    model,
    tokenizer,
    sentences,
    batch_size: int = 64,
    max_length: int = 64,
    device=None,
    show_progress: bool = True,
):
    """
    Encode a list of strings into [N, dim] numpy embeddings.
    """
    if isinstance(sentences, str):
        sentences = [sentences]

    if device is None:
        device = next(model.parameters()).device

    model.eval()
    all_embs = []

    dataloader = DataLoader(sentences, batch_size=batch_size)
    iterator = tqdm(dataloader) if show_progress else dataloader

    with torch.no_grad():
        for batch in iterator:
            enc = tokenizer(
                list(batch),
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            # SentenceEmbedder forward returns normalized embeddings already
            z = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                return_encoder_output=False,
            )  # [B, dim]

            all_embs.append(z.cpu())

    embs = torch.cat(all_embs, dim=0)  # [N, dim]
    return embs.numpy()


In [29]:

class SentenceCorpus:
    """
    Holder for a corpus of sentences with their document and position info.
    Input: list of documents, where each document is list[str] (sentences in order).
    This lets us define 'neighbor' sentences (within a distance window).
    """
    def __init__(self, docs: List[List[str]]):
        self.docs = docs
        # flatten to a list of (doc_id, sent_idx, text)
        self.flat = []
        for d_id, doc in enumerate(docs):
            for s_idx, text in enumerate(doc):
                self.flat.append((d_id, s_idx, text))
        self.N = len(self.flat)

    def get_sentence(self, flat_idx: int) -> str:
        return self.flat[flat_idx][2]

    def neighbor_indices(self, flat_idx: int, max_dist: int = 10) -> List[int]:
        """Return indices of sentences within distance < max_dist in same document (excluding itself)."""
        d_id, s_idx, _ = self.flat[flat_idx]
        doc = self.docs[d_id]
        neighbors = []
        lo = max(0, s_idx - max_dist + 1)
        hi = min(len(doc) - 1, s_idx + max_dist - 1)
        if lo <= hi:
            for j in range(lo, hi + 1):
                if j == s_idx:
                    continue
                # map (d_id, j) back to flat index: we can scan (cheap once) or build index map
                # Let's build mapping once:
            # we'll use a mapping built in constructor
        return []  # unused; dataset uses precomputed mapping

class ContrastiveSentenceDataset(Dataset):
    """
    Dataset returning an anchor index. The collate function constructs multiple text inputs:
      - anchor_text (used twice for dropout augmentations in model forward)
      - neighbor_texts (0..k positives from nearby sentences)
    The collator will produce tokenized batches.
    """
    def __init__(self,
                 docs: List[List[str]],
                 neighbor_window: int = 10,
                 num_neighbors: int = 1,
                 sample_neighbors_prob: float = 1.0):
        """
        docs: list of documents (each is list of sentences)
        neighbor_window: maximum sentence distance to consider neighbor (distance < neighbor_window)
        num_neighbors: number of neighbor positives to sample per anchor (if available)
        sample_neighbors_prob: probability to include neighbor positives (for mixing supervised/unsupervised)
        """
        self.docs = docs
        self.corpus = SentenceCorpus(docs)
        self.num_neighbors = num_neighbors
        self.neighbor_window = neighbor_window
        self.sample_neighbors_prob = sample_neighbors_prob

        # build mapping (doc_id, sent_idx) -> flat index for quick neighbor lookup
        self.doc_index_starts = []
        flat_idx = 0
        for doc in docs:
            self.doc_index_starts.append(flat_idx)
            flat_idx += len(doc)

        # map (doc_id, sent_idx) -> flat_index:
        self.doc_pos_to_flat = {}
        flat = 0
        for d_id, doc in enumerate(docs):
            for s_idx, _ in enumerate(doc):
                self.doc_pos_to_flat[(d_id, s_idx)] = flat
                flat += 1
        self.N = flat

    def __len__(self):
        return self.N

    def sample_neighbors_for_flat_idx(self, flat_idx: int) -> List[int]:
        d_id, s_idx, _ = self.corpus.flat[flat_idx]
        doc = self.docs[d_id]
        lo = max(0, s_idx - (self.neighbor_window - 1))
        hi = min(len(doc) - 1, s_idx + (self.neighbor_window - 1))
        candidates = [j for j in range(lo, hi + 1) if j != s_idx]
        if not candidates:
            return []
        k = min(self.num_neighbors, len(candidates))
        sampled = random.sample(candidates, k)
        flat_sampled = [self.doc_pos_to_flat[(d_id, j)] for j in sampled]
        return flat_sampled

    def __getitem__(self, idx: int):
        """
        Returns an item describing the anchor and indices of positives:
          {
            'anchor_idx': int,
            'anchor_text': str,
            'neighbor_pos_indices': List[int]  # could be empty
          }
        """
        anchor_text = self.corpus.get_sentence(idx)
        neighbor_indices = []
        if random.random() < self.sample_neighbors_prob:
            neighbor_indices = self.sample_neighbors_for_flat_idx(idx)
        return {
            'anchor_idx': idx,
            'anchor_text': anchor_text,
            'neighbor_pos_indices': neighbor_indices
        }

# evaluate

## spearman rank

In [8]:
from scipy.stats import spearmanr

def evaluate_sts_spearman(
    model,
    tokenizer,
    sentences1,
    sentences2,
    scores,
    batch_size: int = 64,
    max_length: int = 64,
    device=None,
):
    assert len(sentences1) == len(sentences2) == len(scores)

    emb1 = encode_sentences(
        model, tokenizer, sentences1,
        batch_size=batch_size, max_length=max_length,
        device=device, show_progress=True
    )
    emb2 = encode_sentences(
        model, tokenizer, sentences2,
        batch_size=batch_size, max_length=max_length,
        device=device, show_progress=True
    )

    cos_sim = (emb1 * emb2).sum(axis=1)  # cosine = dot because normalized

    rho, pval = spearmanr(scores, cos_sim)
    return {
        "spearman_rho": float(rho),
        "p_value": float(pval),
    }


In [26]:
import gzip, csv
from pathlib import Path

sts_path = Path("/content/drive/MyDrive/weights-UML-train/stsbenchmark/stsbenchmark.tsv.gz")  # <- FIX THIS

test_pairs = []  # (sent1, sent2, gold_score)

with gzip.open(sts_path, "rt", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
    for row in reader:
        if row["split"] == "test":
            # original scores are 0–5; Spearman is scale-invariant,
            # but we can normalize to 0–1 like SBERT
            score = float(row["score"]) / 5.0
            test_pairs.append((row["sentence1"], row["sentence2"], score))

print("Loaded STS-B test pairs:", len(test_pairs))


Loaded STS-B test pairs: 1379


In [27]:
from transformers import AutoTokenizer
import torch
import numpy as np
from scipy.stats import spearmanr

device = "cuda" if torch.cuda.is_available() else "cpu"

# must match training
base_model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

def encode_sentences(model, tokenizer, sentences, batch_size=64, max_length=64):
    """
    Returns a (N, d) numpy array of L2-normalized embeddings.
    """
    model.eval()
    all_embs = []

    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch_sents = sentences[i:i+batch_size]
            toks = tokenizer(
                batch_sents,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device)

            emb = model(
                input_ids=toks["input_ids"],
                attention_mask=toks["attention_mask"],
                return_encoder_output=False  # just projected embedding
            )  # (B, d), already normalized in forward()
            all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0).numpy()


In [28]:
# Unpack STS-B test data
sents1, sents2, gold_scores = zip(*test_pairs)
gold_scores = np.array(gold_scores, dtype=np.float32)

# Encode with your trained embedder
emb1 = encode_sentences(model, tokenizer, list(sents1))
emb2 = encode_sentences(model, tokenizer, list(sents2))

# Cosine similarity (embeddings already L2-normalized by SentenceEmbedder)
cos_sim = (emb1 * emb2).sum(axis=1)

# Spearman rank correlation
spearman = spearmanr(cos_sim, gold_scores).correlation
print(f"STS-B test Spearman (cosine): {spearman:.4f}")


STS-B test Spearman (cosine): 0.4802


# alignment & uniformity

In [39]:
import torch

def lalign(x, y, alpha: float = 2.0):
    # x, y: [bsz, d] normalized embeddings
    return (x - y).norm(dim=1).pow(alpha).mean()

def lunif(x, t: float = 2.0):
    # x: [N, d] normalized embeddings
    sq_pdist = torch.pdist(x, p=2).pow(2)
    return sq_pdist.mul(-t).exp().mean().log()


In [40]:
import random

def sample_pos_pairs_from_docs(
    docs,
    num_pairs: int = 10_000,
    neighbor_window: int = 10,
    num_neighbors: int = 1,
    sample_neighbors_prob: float = 1.0,
):
    dataset = ContrastiveSentenceDataset(
        docs=docs,
        neighbor_window=neighbor_window,
        num_neighbors=num_neighbors,
        sample_neighbors_prob=sample_neighbors_prob,
    )

    pairs = []
    indices = list(range(len(dataset)))
    random.shuffle(indices)

    for idx in indices:
        item = dataset[idx]
        anchor_text = item["anchor_text"]
        for flat_idx in item["neighbor_pos_indices"]:
            neighbor_text = dataset.corpus.get_sentence(flat_idx)
            pairs.append((anchor_text, neighbor_text))
            if len(pairs) >= num_pairs:
                return pairs

    return pairs


In [41]:
def eval_alignment_from_docs(
    model,
    tokenizer,
    docs,
    num_pairs: int = 10_000,
    neighbor_window: int = 10,
    alpha: float = 2.0,
    batch_size: int = 64,
    max_length: int = 64,
    device=None,
):
    if device is None:
        device = next(model.parameters()).device

    pos_pairs = sample_pos_pairs_from_docs(
        docs,
        num_pairs=num_pairs,
        neighbor_window=neighbor_window,
        num_neighbors=1,
        sample_neighbors_prob=1.0,
    )
    if not pos_pairs:
        raise ValueError("No positive pairs found in docs.")

    s1 = [p[0] for p in pos_pairs]
    s2 = [p[1] for p in pos_pairs]

    emb1 = encode_sentences(
        model, tokenizer, s1,
        batch_size=batch_size, max_length=max_length,
        device=device, show_progress=True,
    )
    emb2 = encode_sentences(
        model, tokenizer, s2,
        batch_size=batch_size, max_length=max_length,
        device=device, show_progress=True,
    )

    x = torch.from_numpy(emb1)
    y = torch.from_numpy(emb2)

    return lalign(x, y, alpha=alpha).item()


In [42]:
import pandas as pd

def parquet_to_docs(path: str, text_col: str = "text", doc_col: str = "doc_id"):
    """
    Convert parquet with columns [doc_col, text_col] into docs: List[List[str]].
    Each doc is the list of its sentences in order.
    """
    df = pd.read_parquet(path)
    # Make sure we’re sorted by (doc_id, some position) if you have a pos column.
    if "sent_idx" in df.columns:
        df = df.sort_values([doc_col, "sent_idx"])
    else:
        df = df.sort_values(doc_col)

    docs = []
    for _, g in df.groupby(doc_col):
        docs.append(g[text_col].astype(str).tolist())
    return docs


In [43]:
scientific_parquet_test = "/content/drive/MyDrive/weights-UML-train/papers_test.parquet"

docs_scientific_test = parquet_to_docs(
    path=scientific_parquet_test,
    text_col="sentence",      # CHANGE if your column is named differently
    doc_col="doc_id",     # CHANGE if your grouping column is different
)

print("Num scientific docs:", len(docs_scientific_test))
print("Example doc length:", len(docs_scientific_test[0]))

Num scientific docs: 1
Example doc length: 6


In [45]:
scientific_parquet_train = "/content/drive/MyDrive/weights-UML-train/papers_train.parquet"

docs_scientific_train = parquet_to_docs(
    path=scientific_parquet_train,
    text_col="sentence",      # CHANGE if your column is named differently
    doc_col="doc_id",     # CHANGE if your grouping column is different
)

print("Num scientific docs:", len(docs_scientific_train))
print("Example doc length:", len(docs_scientific_train[0]))

Num scientific docs: 1
Example doc length: 26


In [47]:
alignment_scientific_test = eval_alignment_from_docs(
    model,
    tokenizer,
    docs_scientific_test,
    num_pairs=5000,
    neighbor_window=10,
    alpha=2.0,
    batch_size=64,
    max_length=64,
    device=device,
)
print("Alignment (papers, test) =", alignment_scientific_test)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Alignment (papers, test) = 1.2216969728469849


In [46]:
alignment_scientific_train = eval_alignment_from_docs(
    model,
    tokenizer,
    docs_scientific_train,
    num_pairs=5000,
    neighbor_window=10,
    alpha=2.0,
    batch_size=64,
    max_length=64,
    device=device,
)
print("Alignment (papers, train) =", alignment_scientific_train)


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Alignment (papers, train) = 1.3915396928787231


In [48]:
songs_parquet_test = "/content/drive/MyDrive/weights-UML-train/songs_test.parquet"

docs_songs_test = parquet_to_docs(
    path=songs_parquet_test,
    text_col="sentence",      # CHANGE if your column is named differently
    doc_col="doc_id",     # CHANGE if your grouping column is different
)

print("Num songs docs:", len(docs_songs_test))
print("Example doc length:", len(docs_songs_test[0]))

Num songs docs: 1
Example doc length: 4


In [49]:
songs_parquet_train = "/content/drive/MyDrive/weights-UML-train/songs_train.parquet"

docs_songs_train = parquet_to_docs(
    path=songs_parquet_train,
    text_col="sentence",      # CHANGE if your column is named differently
    doc_col="doc_id",     # CHANGE if your grouping column is different
)

print("Num songs docs:", len(docs_songs_train))
print("Example doc length:", len(docs_songs_train[0]))

Num songs docs: 1
Example doc length: 15


In [52]:
alignment_songs_test = eval_alignment_from_docs(
    model,
    tokenizer,
    docs_songs_test,
    num_pairs=5000,
    neighbor_window=10,
    alpha=2.0,
    batch_size=64,
    max_length=64,
    device=device,
)
print("Alignment (songs, test) =", alignment_songs_test)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Alignment (songs, test) = 1.3049876689910889


In [51]:
alignment_songs_train = eval_alignment_from_docs(
    model,
    tokenizer,
    docs_songs_train,
    num_pairs=5000,
    neighbor_window=10,
    alpha=2.0,
    batch_size=64,
    max_length=64,
    device=device,
)
print("Alignment (songs, train) =", alignment_songs_train)


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Alignment (songs, train) = 1.043967604637146


In [53]:
def uniformity_on_sentences(
    model,
    tokenizer,
    sentences,
    t: float = 2.0,
    max_samples: int = 8192,
    batch_size: int = 64,
    max_length: int = 64,
    device=None,
    show_progress: bool = True,
):
    import random
    if len(sentences) > max_samples:
        sentences = random.sample(sentences, max_samples)

    emb = encode_sentences(
        model, tokenizer, sentences,
        batch_size=batch_size, max_length=max_length,
        device=device, show_progress=show_progress
    )
    x = torch.from_numpy(emb)
    return lunif(x, t=t).item()


In [55]:
import pandas as pd

def uniformity_from_parquet(
    path: str,
    text_col: str,
    dataset_name: str,
    t: float = 2.0,
):
    df = pd.read_parquet(path)
    sentences = df[text_col].astype(str).tolist()
    print(f"{dataset_name}: {len(sentences)} sentences")

    u = uniformity_on_sentences(
        model, tokenizer, sentences,
        t=t, max_samples=8192, batch_size=64,
        max_length=64, device=device,
    )
    print(f"Uniformity ({dataset_name}) = {u:.4f}")
    return u


In [56]:
u_songs_test = uniformity_from_parquet(
    "/content/drive/MyDrive/weights-UML-train/songs_test.parquet",
    text_col="sentence",
    dataset_name="songs, test",
)

u_scientific_test = uniformity_from_parquet(
    "/content/drive/MyDrive/weights-UML-train/papers_test.parquet",
    text_col="sentence",
    dataset_name="scientific, test",
)


u_songs_train = uniformity_from_parquet(
    "/content/drive/MyDrive/weights-UML-train/songs_train.parquet",
    text_col="sentence",
    dataset_name="songs, train",
)

u_scientific_train = uniformity_from_parquet(
    "/content/drive/MyDrive/weights-UML-train/papers_train.parquet",
    text_col="sentence",
    dataset_name="scientific, train",
)

songs, test: 4 sentences


  0%|          | 0/1 [00:00<?, ?it/s]

Uniformity (songs, test) = -2.4597
scientific, test: 6 sentences


  0%|          | 0/1 [00:00<?, ?it/s]

Uniformity (scientific, test) = -2.5819
songs, train: 15 sentences


  0%|          | 0/1 [00:00<?, ?it/s]

Uniformity (songs, train) = -2.1511
scientific, train: 26 sentences


  0%|          | 0/1 [00:00<?, ?it/s]

Uniformity (scientific, train) = -2.7964


In [ ]:
# a sanity check

import torch

sample_sents = [
    "This is a test sentence.",
    "Another one.",
    "This is a different sentence.",
    "Totally unrelated text."
]

emb = encode_sentences(model, tokenizer, sample_sents, device=device)
norms = torch.from_numpy(emb).norm(dim=1)
print(norms)  # should be all very close to 1.0


## from same section inside same doc / from same song ?

In [ ]:
import pandas as pd
import random

def within_doc_similarity_stats(
    model,
    tokenizer,
    parquet_path: str,
    text_col: str = "sentence",
    doc_col: str = "doc_id",
    batch_size: int = 64,
    max_length: int = 64,
    max_pairs_per_doc: int = 100,
    max_docs: int = None,      # if you want to cap docs for speed; or None for all
    device=None,
):
    """
    For a parquet with (doc_id, text), compute cosine similarity statistics
    for random sentence pairs sampled *within the same doc*.

    Returns a dict with { 'num_docs', 'num_pairs', 'mean', 'median', 'std', 'min', 'max' }.
    """
    if device is None:
        device = next(model.parameters()).device

    df = pd.read_parquet(parquet_path)
    df = df[[doc_col, text_col]].dropna()

    # Optionally subsample docs for speed
    doc_ids = df[doc_col].unique().tolist()
    if max_docs is not None and len(doc_ids) > max_docs:
        doc_ids = random.sample(doc_ids, max_docs)

    df = df[df[doc_col].isin(doc_ids)].reset_index(drop=True)

    print(f"Loaded {len(df)} sentences across {len(doc_ids)} docs from {parquet_path}")

    # Encode all sentences once
    sentences = df[text_col].astype(str).tolist()
    embeddings = encode_sentences(
        model,
        tokenizer,
        sentences,
        batch_size=batch_size,
        max_length=max_length,
        device=device,
        show_progress=True,
    )  # [N, dim]

    # Pre-map row index -> embedding
    emb_t = torch.from_numpy(embeddings)  # [N, dim], still normalized

    # Build index lists per doc
    doc_to_indices = {}
    for idx, doc_id in enumerate(df[doc_col].tolist()):
        doc_to_indices.setdefault(doc_id, []).append(idx)

    cos_sims = []

    for doc_id, idxs in doc_to_indices.items():
        if len(idxs) < 2:
            continue  # need at least 2 sentences to make a pair

        # Sample pairs within this doc
        # If doc is small, use all pairs; if large, subsample.
        if len(idxs) * (len(idxs) - 1) // 2 <= max_pairs_per_doc:
            # all unordered pairs
            for i in range(len(idxs)):
                for j in range(i + 1, len(idxs)):
                    a = emb_t[idxs[i]]
                    b = emb_t[idxs[j]]
                    cos = torch.dot(a, b).item()  # cosine because normalized
                    cos_sims.append(cos)
        else:
            # random sampling of pairs
            for _ in range(max_pairs_per_doc):
                i, j = random.sample(idxs, 2)
                a = emb_t[i]
                b = emb_t[j]
                cos = torch.dot(a, b).item()
                cos_sims.append(cos)

    if not cos_sims:
        raise ValueError("No within-doc pairs found (check doc_id / text columns).")

    cos_sims = np.array(cos_sims, dtype=np.float32)
    stats = {
        "num_docs": len(doc_to_indices),
        "num_pairs": int(len(cos_sims)),
        "mean": float(cos_sims.mean()),
        "median": float(np.median(cos_sims)),
        "std": float(cos_sims.std()),
        "min": float(cos_sims.min()),
        "max": float(cos_sims.max()),
    }
    return stats

In [ ]:
songs_train_path = "/content/drive/MyDrive/weights-UML-train/songs_train.parquet"
songs_test_path  = "/content/drive/MyDrive/weights-UML-train/songs_test.parquet"
papers_train_path = "/content/drive/MyDrive/weights-UML-train/papers_train.parquet"
papers_test_path  = "/content/drive/MyDrive/weights-UML-train/papers_test.parquet"

In [ ]:
subsets = {
    "songs_train": songs_train_path,
    "songs_test": songs_test_path,
    "papers_train": papers_train_path,
    "papers_test": papers_test_path,
}

results = {}

for name, path in subsets.items():
    print(f"\n=== {name} ===")
    stats = within_doc_similarity_stats(
        model,
        tokenizer,
        parquet_path=path,
        text_col="sentence",      # change if your column names differ
        doc_col="doc_id",
        batch_size=64,
        max_length=64,
        max_pairs_per_doc=100,
        max_docs=None,        # or e.g. 500 if huge
        device=device,
    )
    results[name] = stats
    print(
        f"{name}: num_docs={stats['num_docs']}, "
        f"num_pairs={stats['num_pairs']}, "
        f"mean_cos={stats['mean']:.4f}, "
        f"median={stats['median']:.4f}, "
        f"std={stats['std']:.4f}, "
        f"min={stats['min']:.4f}, "
        f"max={stats['max']:.4f}"
    )